# C · Direct, static and temporal-refinement benchmarks

This experiment establishes whether the proposed restoration improves on practical alternatives. Direct models learn coordinate correction end to end. The static model removes cross-frame coordinates while retaining full-window confidence, availability and time. The temporal refiner adapts a learned trajectory-refinement architecture to the same body-12 inputs.

Read after notebooks 00–06. This notebook uses the prepared bundle and saved evaluation from that same run; the internal recipe group is `P`.


In [ ]:
from pathlib import Path
import json, os, sys

# Find the checkout/release from the notebook's working directory.
ROOT = Path(os.environ.get('GF_ROOT', Path.cwd())).resolve()
while not (ROOT / 'src/gavd6_sjepa').is_dir() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
assert (ROOT / 'src/gavd6_sjepa').is_dir(), 'Open this notebook from the GAVD6 checkout or release.'
sys.path.insert(0, str(ROOT / 'notebooks/gait_fidelity'))
sys.path.insert(0, str(ROOT / 'src'))
from tutorial_helpers import configure, preview_images
study = configure(ROOT)


## Inspect the exact recipe cells

The direct base and direct paired-change recipes isolate the added movement objective within one trainable architecture. Static and temporal-refiner base recipes provide practical benchmarks but do not have every loss-matched variant. Also read unchanged tracks, training-only offsets and affine calibration, and fixed temporal filters from the common evaluation.


In [ ]:
import pandas as pd
plan = study.artifact('plan.json')
group_recipes = [r for r in plan['recipes'] if r['group'] == 'P']
expected_count = 4 if plan.get('experiment_set', 'full') == 'full' else {'M': 4, 'T': 0, 'P': 2, 'I': 4, 'L': 0}['P']
assert len(group_recipes) == expected_count, 'Saved plan differs from the selected experiment set.'
recipe_ids = {r['recipe_id'] for r in group_recipes}
display(pd.DataFrame(group_recipes))
print('Final models:', len(group_recipes) * len(plan['seeds']), 'Seeds:', plan['seeds'])
if not group_recipes:
    print('This group is outside the saved core protocol. Its worked examples are educational; no results are implied.')


## Fit the spatial controls using training people only

In normalized coordinates, the joint offset is the weighted mean training
residual $b_j=\sum w(y_j-x_j)/\sum w$. The affine control fits
$W_j=(D_j^T W D_j+\Lambda)^{-1}D_j^T W(y_j-x_j)$, where each design row is
$D_j=[x_j^x,x_j^y,1]$ and $\Lambda=\operatorname{diag}(0.001,0.001,0)$.
The intercept is unpenalized; `solve` evaluates the expression without
explicitly forming an inverse. A training observation needs both an
observed input joint and a valid reference. Every development extractor
receives the same fitted correction. Source runs give equal mass to people,
then raw motions within a person, windows within a motion, and variants
within a window; available frames within a joint share the window weight.
The software fixture retains its legacy equal-frame calibration. The
sufficient statistics below are accumulated in bounded batches, so the
full training arrays need not be copied into RAM.

Notebook 01 derives the input-only normalization. Here its transform is
reused to isolate the fitting calculation. The correction is converted
back to pixels with the development input's own origin and scale. Missing
input joints remain missing in these two spatial controls.


In [ ]:
import numpy as np
from gavd6_sjepa.research_directions.gait_fidelity.data import load_dataset
from gavd6_sjepa.research_directions.gait_fidelity.training import normalize_inputs
data = load_dataset(study.bundle_path())
train, development = data.subset('train'), data.subset('development')
train_people = {r['canonical_person_id'] for r in train.records}
assert train_people.isdisjoint(r['canonical_person_id'] for r in development.records)
config = study.artifact('config.json')
balanced = config['training'].get('sampling') == 'person_motion'
groups = {}
for i, row in enumerate(train.records):
    groups.setdefault(row['canonical_person_id'], {}).setdefault(row['motion_hash'], {}).setdefault(row['source_family_id'], []).append(i)
row_weights = np.ones(len(train.records))
if balanced:
    for person in groups.values():
        for motion in person.values():
            for rows in motion.values():
                row_weights[rows] = 1 / (len(groups) * len(person) * len(motion) * len(rows))
gram, cross = np.zeros((12,3,3)), np.zeros((12,3,2))
total, residual_sum = np.zeros(12), np.zeros((12,2))
for start in range(0, len(train.records), 256):
    sl = slice(start, start + 256)
    raw = {k: v[sl] for k,v in train.inputs.items()}
    x, transform = normalize_inputs(raw)
    y = transform.apply(train.targets['xy'][sl])
    for joint in range(12):
        keep = raw['observed'][:,:,joint] & train.targets['valid'][sl,:,joint]
        weight = row_weights[sl] / np.maximum(keep.sum(1),1) if balanced else row_weights[sl]
        weight = np.broadcast_to(weight[:,None], keep.shape)[keep]
        source = x['xy'][:,:,joint][keep]
        residual = y[:,:,joint][keep] - source
        design = np.column_stack([source, np.ones(len(source))])
        gram[joint] += design.T @ (design * weight[:,None])
        cross[joint] += design.T @ (residual * weight[:,None])
        residual_sum[joint] += (residual * weight[:,None]).sum(0)
        total[joint] += weight.sum()
assert np.all(total > 0)
offset = residual_sum / total[:,None]
affine = np.stack([np.linalg.solve(g + np.diag([.001,.001,0.]), c) for g,c in zip(gram,cross)])
saved = study.artifact('evaluation/calibration.json')
np.testing.assert_allclose(offset, saved['offset'], rtol=1e-6, atol=1e-8)
np.testing.assert_allclose(affine, saved['affine'], rtol=1e-6, atol=1e-8)
preview = {k: v[:4] for k,v in development.inputs.items()}
z, development_transform = normalize_inputs(preview)
design = np.concatenate([z['xy'], np.ones((*z['observed'].shape, 1))], axis=-1)
normalized = z['xy'] + np.einsum('ntjk,jkc->ntjc', design, affine)
restored = np.where(z['observed'][..., None], development_transform.invert(normalized), np.nan)
exported = np.load(study.artifact('evaluation/joint_affine.npy'), mmap_mode='r', allow_pickle=False)
np.testing.assert_allclose(restored, exported[:4], rtol=1e-6, atol=1e-6, equal_nan=True)
print('Training-only coefficients and development predictions match the retained calibration.')


## Reproduce interpolation and temporal smoothing on one track

The fixed filters first linearly interpolate observed positions in physical
time, extending the first and last observed values to the crop edges.
They then apply centered triangular weights. Strength 1 uses $[1,2,1]/4$;
strength 2 uses $[1,2,3,2,1]/9$. Strength 0 performs interpolation alone.
A wholly missing joint remains missing. These are offline baselines because
a centered filter uses frames on both sides of each output time.


In [ ]:
import matplotlib.pyplot as plt
from io import BytesIO
from IPython.display import Image, display
from gavd6_sjepa.research_directions.synthetic_training_v2.data import filter_tracks
one = {k: v[:1].copy() for k, v in development.inputs.items()}
times = one['timestamps'][0]
all_filtered = {}
for strength in [0, 1, 2]:
    result = np.full_like(one['xy'], np.nan)
    for joint in range(12):
        keep = one['observed'][0, :, joint]
        if not keep.any():
            continue
        for axis in range(2):
            values = np.interp(times, times[keep], one['xy'][0, keep, joint, axis])
            if strength:
                weights = np.r_[np.arange(1, strength + 2), np.arange(strength, 0, -1)]
                weights = weights / weights.sum()
                values = np.convolve(np.pad(values, (strength, strength), mode='edge'), weights, mode='valid')
            result[0, :, joint, axis] = values
    np.testing.assert_allclose(result, filter_tracks(one, strength), equal_nan=True)
    all_filtered[strength] = result
fig, ax = plt.subplots(figsize=(9, 3.5), layout='constrained')
ax.plot(times, one['xy'][0, :, 10, 0], '.', label='Estimated left ankle')
ax.plot(times, development.targets['xy'][0, :, 10, 0], color='black', label='Projected reference')
for strength, values in all_filtered.items():
    ax.plot(times, values[0, :, 10, 0], label=f'Filter {strength}')
label = 'Software fixture' if study.fixture else 'Source development example'
ax.set(xlabel='Time (s)', ylabel='Horizontal position (pixels)',
       title=label + ': first record selected by metadata order')
ax.legend(fontsize=8)
buffer = BytesIO()
fig.savefig(buffer, format='png', dpi=120, bbox_inches='tight')
display(Image(data=buffer.getvalue()))
plt.close(fig)


An offset preserves pixel displacement within a window; the affine and
temporal controls can alter it. The static neural benchmark receives
coordinates from the current frame together with an encoded full-window
vector of confidence, availability and relative time. Its normalization
also uses the whole input window. It removes cross-frame coordinate
trajectories while retaining this auxiliary information. The temporal
refiner is a study-specific adaptation, not a reproduction with published
pretrained weights. Read both position and movement errors in notebook 05.


## Follow the shared dependencies

This tutorial inspects the existing central queue. It does not launch a separate copy of its group: that would duplicate pretraining and break the global budget. Notebook 04 launches all groups, and this table identifies the phases that belong to the present comparison.


In [ ]:
final_phases = [p for p in plan['phases'] if p['phase'] != 'pretrain' and p['recipe']['recipe_id'] in recipe_ids]
parent_ids = {parent for p in final_phases for parent in p['depends_on'] if parent != 'prepare'}
selected = [p for p in plan['phases'] if p in final_phases or p['phase_id'] in parent_ids]
display(pd.DataFrame([{'phase_id': p['phase_id'], 'phase': p['phase'], 'seed': p['seed'],
                      'depends_on': ', '.join(p['depends_on'])} for p in selected]))


## Inspect completed checkpoints and learning histories

Each completed phase links its retained checkpoint, history and predictions to its source identity. Missing phases are reported as pending; a checkpoint from another study is not substituted.


In [ ]:
ledger_path = study.work / 'ledger.json'
completed = json.loads(ledger_path.read_text()).get('completed', {}) if ledger_path.exists() else {}
rows = []
for phase in selected:
    saved = completed.get(phase['phase_id'])
    result = saved.get('result', {}) if saved else {}
    rows.append({'phase_id': phase['phase_id'], 'status': 'complete' if saved else 'pending',
                 'checkpoint': result.get('checkpoint'), 'predictions': result.get('predictions')})
display(pd.DataFrame(rows))
history_candidates = []
for row in rows:
    if row['checkpoint']:
        history_path = Path(row['checkpoint']).parent / 'history.json'
        if history_path.exists(): history_candidates.append(history_path)
if history_candidates:
    history_path = history_candidates[0]
    print('First declared completed history:', history_path)
    display(pd.DataFrame(json.loads(history_path.read_text())))
else:
    print('No completed histories yet. Run or resume the central queue from notebook 04.')


## Read group results on the common population

State the temporal refiner's adaptation explicitly rather than treating it as a reproduction with the original authors' weights. Lower coordinate error and lower movement error can favor different methods; use the full paired population, support and failures before choosing a scientific conclusion.


In [ ]:
person_path = study.work / 'evaluation/per-person.csv'
if person_path.exists():
    people = pd.read_csv(person_path)
    display(people.loc[people['method'].isin(recipe_ids)])
    coverage_path = study.work / 'evaluation/coverage.csv'
    if coverage_path.exists():
        coverage = pd.read_csv(coverage_path)
        display(coverage.loc[coverage['method'].isin(recipe_ids)])
else:
    print('Evaluation is pending. These recipe cards do not fabricate or extrapolate results.')


Read the matching controls from the other experiment tutorials before attribution. Full evaluation and numerical reconstruction are covered by notebooks 05 and 06.
